In [ ]:
# Data loading - keep paths in notebook for easy switching
noisy_train_path = "/gpfs/data/rinberglab/vivek/alt_train_glom_data/alt_train_glom_data.pt"
noisy_test_path = "/gpfs/data/rinberglab/vivek/alt_train_glom_data/alt_train_glom_data.pt"

# Use our clean function from src/data_processing.py
from src.data_processing import load_glomeruli_data
train_data, train_labels, test_data, test_labels = load_glomeruli_data(
    noisy_train_path, noisy_test_path
)



In [ ]:
# Import your exact function
from src.utils import calculate_dataset_stats

# Keep your exact same usage code
train_stats = calculate_dataset_stats(noisy_train_data)
print("Train Dataset Statistics:")
for key, value in train_stats.items():
    print(f"{key.capitalize()}: {value}")

test_stats = calculate_dataset_stats(noisy_test_data)
print("\nTest Dataset Statistics:")
for key, value in test_stats.items():
    print(f"{key.capitalize()}: {value}")
    

In [ ]:
# Glomeruli Denoising - Main Notebook
# Clean organized version of the original notebook

# Cell 1: Imports
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.functional import pairwise_distance
from torch.optim.lr_scheduler import StepLR
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix
import seaborn as sns
from torch.utils.data import Dataset
import random
import os
import re
import json
import cv2
import wandb

# Import our custom functions
from src.data_processing import load_glomeruli_data
from src.dataset import GlomeruliContrastiveDataset
from src.utils import calculate_dataset_stats

print(torch.__version__)
print(torchvision.__version__)

# Cell 2: Data Loading
# Define the paths to save the datasets
noisy_train_path = "/gpfs/data/rinberglab/vivek/alt_train_glom_data/alt_train_glom_data.pt"
noisy_test_path = "/gpfs/data/rinberglab/vivek/alt_train_glom_data/alt_train_glom_data.pt"

# Load data using our function
train_data, train_labels, test_data, test_labels = load_glomeruli_data(
    noisy_train_path, noisy_test_path
)

# Cell 3: Dataset Statistics
# Calculate statistics for train data
train_stats = calculate_dataset_stats(train_data)
print("Train Dataset Statistics:")
for key, value in train_stats.items():
    print(f"{key.capitalize()}: {value}")

# Calculate statistics for test data
test_stats = calculate_dataset_stats(test_data)
print("\nTest Dataset Statistics:")
for key, value in test_stats.items():
    print(f"{key.capitalize()}: {value}")




In [ ]:
# Assuming noisy_train_data and noisy_train_labels are your glomeruli activation data and labels

noisy_glomeruli_dataset = GlomeruliContrastiveDataset(noisy_train_data, noisy_train_labels, is_training=True)
train_loader = DataLoader(noisy_glomeruli_dataset, batch_size=32, shuffle=True)

# Assuming noisy_train_data and noisy_train_labels are your glomeruli activation data and labels

noisy_glomeruli_dataset_test = GlomeruliContrastiveDataset(noisy_test_data, noisy_test_labels, is_training=False)
test_loader = DataLoader(noisy_glomeruli_dataset_test, batch_size=32, shuffle=True)


In [ ]:
# Cell 5: Dataset Sanity Check
from src.utils import dataset_sanity_check

# Test the dataset
dataset_sanity_check(train_dataset, sample_index=15)

In [ ]:
# Cell 6: Visualize Contrastive Samples
from src.utils import visualize_contrastive_samples

# Visualize samples
visualize_contrastive_samples(train_dataset, sample_index=15)

In [ ]:
# Cell 7: Model Definition
from src.model import get_model

# Create model with descriptive name
model = get_model("simple_cnn")

print(f"Model: GlomeruliDenoiserCNN")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test model shape
test_input = torch.randn(1, 1, 256, 256)  # Batch of 1, single channel, 256x256
test_output = model(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {test_output.shape}")